[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maurilio97-P/6d-pose-vision-workshop/blob/main/part_5_aruco/12_generate_aruco.ipynb)

# Notebook 12 — Generating ArUco Markers

**Part 5: ArUco Markers** | Estimated time: 30 min

---

In the previous notebook we learned the *theory* behind ArUco: binary grids, Hamming distance, dictionaries.  
Now we get practical — we generate real marker images you can print and use.

```
Pipeline position:

  [Generate marker]  →  Print/display  →  Detect  →  Pose estimate
       ← YOU ARE HERE
```

## Recommended Watch

> Watch this **before** generating your marker set.

| Title | Link | Duration |
|---|---|---|
| **Generate ArUco Markers for Detection and Pose Estimation with OpenCV** | [▶ Watch](https://youtu.be/sg1bVJBjbng?si=E6pw53-D06sHjfrg) | ~10 min |

---

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install opencv-contrib-python matplotlib numpy -q
    print('Running in Google Colab — packages installed')
else:
    print('Running locally — make sure your venv is activated')

## Learning Objectives

By the end of this notebook you will be able to:

- Generate a single ArUco marker image with OpenCV
- Save markers as PNG files ready to print
- Batch-generate a full marker set in a loop
- Understand the new vs. old OpenCV ArUco API
- Apply practical printing tips for reliable detection

## 1. Imports

The `cv2.aruco` module is part of `opencv-contrib-python` — **not** the standard `opencv-python`. If the import fails, check that you installed the contrib package:

```bash
pip install opencv-contrib-python   # ← this one, not opencv-python
```

If you're on Colab, the setup cell above handles this automatically.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

print(f'OpenCV version: {cv2.__version__}')

# Helper: display images inline in notebooks
def show_gray(img, title='', figsize=(4, 4)):
    plt.figure(figsize=figsize)
    plt.imshow(img, cmap='gray', vmin=0, vmax=255)
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

def show_many(imgs, titles, ncols=4, cell_size=3):
    nrows = (len(imgs) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(cell_size * ncols, cell_size * nrows))
    axes = np.array(axes).flatten()
    for ax, img, title in zip(axes, imgs, titles):
        ax.imshow(img, cmap='gray', vmin=0, vmax=255)
        ax.set_title(title, fontsize=9)
        ax.axis('off')
    for ax in axes[len(imgs):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 2. Old API vs. New API — A Quick Note

OpenCV 4.7 introduced a **new ArUco API**. You will see both styles in tutorials online.

| Task | Old API (≤ 4.6) | New API (≥ 4.7) |
|---|---|---|
| Get dictionary | `cv2.aruco.Dictionary_get(...)` | `cv2.aruco.getPredefinedDictionary(...)` |
| Draw/generate marker | `cv2.aruco.drawMarker(...)` | `cv2.aruco.generateImageMarker(...)` |
| Detector params | `cv2.aruco.DetectorParameters_create()` | `cv2.aruco.DetectorParameters()` |
| Detect markers | `cv2.aruco.detectMarkers(...)` | `ArucoDetector.detectMarkers(...)` |

**In this course we use the new API.** The code below includes a compatibility shim if you have an older version.

In [ ]:
# Detect which API version is available
def get_aruco_dict(dict_id):
    """Get ArUco dictionary — works with both old and new OpenCV API."""
    try:
        # New API (OpenCV >= 4.7)
        return cv2.aruco.getPredefinedDictionary(dict_id)
    except AttributeError:
        # Old API fallback
        return cv2.aruco.Dictionary_get(dict_id)

def generate_marker(aruco_dict, marker_id, size_px):
    """Generate a marker image — works with both old and new API."""
    try:
        # New API (OpenCV >= 4.7)
        return cv2.aruco.generateImageMarker(aruco_dict, marker_id, size_px)
    except AttributeError:
        # Old API fallback
        img = np.zeros((size_px, size_px, 1), dtype='uint8')
        cv2.aruco.drawMarker(aruco_dict, marker_id, size_px, img, 1)
        return img.squeeze()

print('API compatibility shim ready.')

## 3. Generating Your First Marker

Three things you need to decide:

1. **Which dictionary?** — defines the grid size and max number of unique IDs  
2. **Which ID?** — must be in range `[0, dict_size - 1]`  
3. **How many pixels?** — affects print size; 500–1000 px is good for printing

> **Remember:** When you later *detect* this marker, you MUST use the same dictionary. If you generate with `DICT_4X4_50` but detect with `DICT_5X5_100`, you will get no detections.

In [ ]:
# --- Parameters ---
DICT_ID   = cv2.aruco.DICT_4X4_50  # 4x4 grid, 50 unique markers (IDs 0-49)
MARKER_ID = 0                       # Which marker to generate
SIZE_PX   = 600                     # Pixel size of output image

# --- Generate ---
aruco_dict = get_aruco_dict(DICT_ID)
marker_img = generate_marker(aruco_dict, MARKER_ID, SIZE_PX)

print(f'Generated marker: ID={MARKER_ID}, shape={marker_img.shape}, dtype={marker_img.dtype}')

# --- Display ---
show_gray(marker_img, f'DICT_4X4_50 — Marker ID {MARKER_ID} ({SIZE_PX}×{SIZE_PX} px)')

### Anatomy of a Generated Marker

```
┌─────────────────────────────────┐
│ ██████████████████████████████  │
│ ██                          ██  │  ← BLACK BORDER (1 cell wide)
│ ██  ■  □  ■  □              ██  │     tells detector: "marker here"
│ ██  □  ■  □  ■              ██  │
│ ██  ■  □  ■  □              ██  │  ← DATA CELLS (4×4 for DICT_4X4)
│ ██  □  □  ■  □              ██  │     encode the unique ID
│ ██████████████████████████████  │
└─────────────────────────────────┘
```

- Outer ring = **always black** — provides contrast, enables corner detection  
- Inner grid = **unique binary pattern** — encodes the ID  
- The pattern also encodes orientation so the detector knows which corner is which

## 4. Saving Markers to PNG

PNG is preferred over JPEG for markers because:

- **PNG is lossless** — JPEG introduces compression artifacts that blur cell edges
- Sharp black/white boundaries are critical for reliable detection
- Even slight blurring can corrupt bit values → wrong ID detected

In [ ]:
# Create output directory
OUTPUT_DIR = '../assets/aruco_markers'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save a single marker
save_path = os.path.join(OUTPUT_DIR, f'4x4_id{MARKER_ID:03d}.png')
success = cv2.imwrite(save_path, marker_img)

if success:
    print(f'Saved: {save_path}')
    file_size_kb = os.path.getsize(save_path) / 1024
    print(f'File size: {file_size_kb:.1f} KB')
else:
    print('ERROR: Could not save marker — check directory path')

## 5. Batch Generation — Generate Multiple Markers

In practice you often need a set of markers — for example, one per station in a factory.  
A simple loop handles this cleanly.

In [ ]:
# The batch marker generator also lives in scripts/generate_aruco_markers.py
# Run it from the repo root in a terminal:
#
#   python scripts/generate_aruco_markers.py
#
# Optional arguments:
#   --dict    NAME    ArUco dictionary (default: DICT_4X4_50)
#   --ids     LIST    IDs to generate: '0 1 2' or range '0-9' (default: 0-9)
#   --size    INT     image size in pixels (default: 600)
#   --border  INT     white border padding (default: 60)
#   --out     PATH    output directory (default: assets/aruco_markers)
#
# Examples:
#   python scripts/generate_aruco_markers.py --ids 0-4 --size 800
#   python scripts/generate_aruco_markers.py --dict DICT_5X5_100 --ids 0 1 2
#
# Prints physical size at 300 DPI so you know how big to print.

# ---- Below: interactive notebook version for experimenting ----
DICT_ID    = cv2.aruco.DICT_4X4_50
IDS_TO_GEN = list(range(10))  # IDs 0 through 9
SIZE_PX    = 600

aruco_dict = get_aruco_dict(DICT_ID)
generated = []

for mid in IDS_TO_GEN:
    img = generate_marker(aruco_dict, mid, SIZE_PX)
    path = os.path.join(OUTPUT_DIR, f'4x4_id{mid:03d}.png')
    cv2.imwrite(path, img)
    generated.append(img)
    print(f'  ID {mid:2d} → saved to {path}')

print(f'\nGenerated {len(IDS_TO_GEN)} markers in {OUTPUT_DIR}/')

In [ ]:
# Visualize all generated markers
titles = [f'ID {i}' for i in IDS_TO_GEN]
# Resize to smaller thumbnails for display
thumbs = [cv2.resize(img, (150, 150)) for img in generated]
show_many(thumbs, titles, ncols=5)

## 6. Comparing Different Dictionaries

The same ID (e.g., ID=5) looks completely different across dictionaries because the bit patterns are independently designed for each family.

In [ ]:
dictionaries = [
    (cv2.aruco.DICT_4X4_50,  'DICT_4X4_50'),
    (cv2.aruco.DICT_5X5_100, 'DICT_5X5_100'),
    (cv2.aruco.DICT_6X6_250, 'DICT_6X6_250'),
    (cv2.aruco.DICT_7X7_1000,'DICT_7X7_1000'),
]

COMPARE_ID = 5
imgs, titles = [], []

for dict_id, name in dictionaries:
    d = get_aruco_dict(dict_id)
    img = generate_marker(d, COMPARE_ID, 300)
    imgs.append(img)
    titles.append(f'{name}\nID={COMPARE_ID}')

show_many(imgs, titles, ncols=4, cell_size=3.5)
print('Same ID, completely different bit patterns!')
print('⚠️  You must use the SAME dictionary for generation AND detection.')

## 7. Printing Tips for Reliable Detection

Generating the PNG is the easy part. Printing it correctly is where things can go wrong.

### Resolution and Physical Size

Most printers print at 300 DPI (dots per inch). To calculate physical print size:

$$\text{print size (inch)} = \frac{\text{pixel size}}{\text{printer DPI}}$$

| Pixel size | @ 300 DPI | @ 150 DPI |
|---|---|---|
| 300 px | 1.0 inch (2.5 cm) | 2.0 inch (5 cm) |
| 600 px | 2.0 inch (5 cm) | 4.0 inch (10 cm) |
| 1200 px | 4.0 inch (10 cm) | 8.0 inch (20 cm) |

**Recommendation for robotics:** Print at 10–15 cm physical size for detection at 1–3 m range.

### Checklist for Print Quality

| ✅ Do | ❌ Don't |
|---|---|
| Use **matte paper** | Use glossy paper (glare kills detection) |
| Print pure **black & white** | Print in color mode (can shift gray levels) |
| Mount **flat** on a rigid surface | Let marker curl or wrinkle |
| Leave a **white border** around marker | Crop marker flush to edge |
| **Laminate** for outdoor use | Leave exposed to humidity |
| Verify print with `cv2.QRCodeDetector` test | Assume print is fine without testing |

In [ ]:
def print_size_report(pixel_size, dpi=300):
    """Calculate physical print dimensions at given DPI."""
    inches = pixel_size / dpi
    cm = inches * 2.54
    print(f'Pixel size: {pixel_size} px')
    print(f'At {dpi} DPI: {inches:.2f}" × {inches:.2f}" = {cm:.1f} cm × {cm:.1f} cm')
    print()

print('=== Print Size Calculator ===')
for px in [300, 600, 900, 1200]:
    print_size_report(px, dpi=300)

## 8. Adding a White Border (Recommended)

The ArUco standard requires a white margin around the marker (called **quiet zone** in barcode terminology).  
OpenCV's `generateImageMarker` includes a thin white border, but you may want to add extra padding.

In [ ]:
def add_white_border(marker_img, border_px=40):
    """Add white padding around a marker image."""
    h, w = marker_img.shape[:2]
    # Create white canvas
    canvas = np.ones((h + 2*border_px, w + 2*border_px), dtype=np.uint8) * 255
    # Place marker in center
    canvas[border_px:border_px+h, border_px:border_px+w] = marker_img
    return canvas

# Generate and add border
aruco_dict = get_aruco_dict(cv2.aruco.DICT_4X4_50)
raw_marker = generate_marker(aruco_dict, 0, 500)
bordered_marker = add_white_border(raw_marker, border_px=60)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(raw_marker, cmap='gray', vmin=0, vmax=255)
axes[0].set_title(f'Raw marker\n{raw_marker.shape[0]}×{raw_marker.shape[1]} px')
axes[0].axis('off')
axes[1].imshow(bordered_marker, cmap='gray', vmin=0, vmax=255)
axes[1].set_title(f'With white border (+60 px)\n{bordered_marker.shape[0]}×{bordered_marker.shape[1]} px')
axes[1].axis('off')
plt.suptitle('Adding a white border improves detection reliability', fontweight='bold')
plt.tight_layout()
plt.show()

# Save bordered version
cv2.imwrite(os.path.join(OUTPUT_DIR, '4x4_id000_bordered.png'), bordered_marker)
print('Bordered marker saved.')

## 9. Practical: Generate a Station Marker Set

Here's a realistic robotics scenario: a mobile robot navigates to three fixed stations.  
Each station gets a unique ArUco ID so the robot knows which station it's approaching.

```
Factory Layout:

  [Charging Station]    [Conveyor A]    [Conveyor B]
     ArUco ID=0          ArUco ID=1      ArUco ID=2
```

In [ ]:
STATION_MARKERS = [
    {'id': 0, 'name': 'Charging Station', 'color_label': 'Green'},
    {'id': 1, 'name': 'Conveyor A',       'color_label': 'Blue'},
    {'id': 2, 'name': 'Conveyor B',       'color_label': 'Red'},
]

PHYSICAL_SIZE_M = 0.15  # 15 cm markers
SIZE_PX = 800

aruco_dict = get_aruco_dict(cv2.aruco.DICT_4X4_50)

station_imgs = []
station_titles = []

for station in STATION_MARKERS:
    marker = generate_marker(aruco_dict, station['id'], SIZE_PX)
    bordered = add_white_border(marker, border_px=60)
    
    path = os.path.join(OUTPUT_DIR, f"station_{station['name'].replace(' ', '_').lower()}_id{station['id']}.png")
    cv2.imwrite(path, bordered)
    
    station_imgs.append(bordered)
    station_titles.append(f"{station['name']}\nID={station['id']}")
    
    print(f"  ✓ {station['name']} (ID={station['id']}) → {path}")

print(f'\nAll station markers saved to: {OUTPUT_DIR}/')
print(f'Physical size when printed at 300 DPI: ~{SIZE_PX/300*2.54:.0f} cm + border')

show_many(station_imgs, station_titles, ncols=3, cell_size=4)

## Recap

| Concept | Key Point |
|---|---|
| `getPredefinedDictionary()` | Loads a dictionary by ID (new API) |
| `generateImageMarker()` | Renders a marker to a NumPy array (new API) |
| Save format | Always **PNG** — lossless, sharp edges |
| Dictionary match | Generation and detection **must** use the same dict |
| Physical size | `pixel_size / dpi` gives print size in inches |
| White border | Adds quiet zone — improves corner detection |
| Matte paper | Prevents glare — significant impact on detection |

**Next notebook:** We detect these markers in images and video streams.

---
## Exercises

In [ ]:
# ============================================================
# EXERCISE 1: Generate a 6x6 marker
# ============================================================
# Generate marker ID=42 from DICT_6X6_250 at 700px size.
# Display it and save it to the OUTPUT_DIR.
# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# d = get_aruco_dict(cv2.aruco.DICT_6X6_250)
# img = generate_marker(d, 42, 700)
# show_gray(img, 'DICT_6X6_250 — ID 42')
# cv2.imwrite(os.path.join(OUTPUT_DIR, '6x6_id042.png'), img)
# print('Saved!')

In [ ]:
# ============================================================
# EXERCISE 2: Batch generate and compute print sizes
# ============================================================
# Generate markers ID 0-5 from DICT_5X5_100 at 600px.
# For each marker, print its physical size at 300 DPI in cm.
# Display all 6 in a grid.
# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# d = get_aruco_dict(cv2.aruco.DICT_5X5_100)
# imgs, titles = [], []
# SIZE = 600
# for i in range(6):
#     img = generate_marker(d, i, SIZE)
#     cm = SIZE / 300 * 2.54
#     print(f'ID {i}: {cm:.1f} cm × {cm:.1f} cm at 300 DPI')
#     imgs.append(img)
#     titles.append(f'5x5 ID={i}')
# show_many(imgs, titles, ncols=3)

In [ ]:
# ============================================================
# EXERCISE 3: Create a print sheet
# ============================================================
# Create a single 1200×1200 white image and tile 4 markers
# (ID 0-3, each 400px) in a 2×2 grid. Save as 'marker_sheet.png'.
# This simulates a printable sheet with multiple markers.
# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# d = get_aruco_dict(cv2.aruco.DICT_4X4_50)
# sheet = np.ones((1200, 1200), dtype=np.uint8) * 255  # white
# for i, mid in enumerate([0, 1, 2, 3]):
#     row, col = divmod(i, 2)
#     m = generate_marker(d, mid, 500)
#     m = cv2.resize(m, (500, 500))
#     r0 = row * 600 + 50
#     c0 = col * 600 + 50
#     sheet[r0:r0+500, c0:c0+500] = m
# show_gray(sheet, '2×2 marker sheet')
# cv2.imwrite(os.path.join(OUTPUT_DIR, 'marker_sheet.png'), sheet)
# print('Sheet saved!')